In [ ]:
from pynq import Overlay
from pynq import MMIO
import numpy as np
import time

import cv2

In [ ]:
image = cv2.imread("Penguin.jpg",cv2.IMREAD_GRAYSCALE)
image = cv2.resize(image,(102,102))

input_data = image.flatten()

In [ ]:
class PynqTestDriver:
    def __init__(self, bitfile_path):
        self.hw = Overlay(bitfile_path)
        self.csr = self.hw.csr_0.mmio.array
        self.imem = self.hw.IMEM_ctrlr.mmio.array
        self.omem = self.hw.OMEM_ctrlr.mmio.array
        
    def get_output(self, length):
        return self.omem[0:length]

    def generate_img(self, save_path = "result_img.jpg"):
        img = self.result.reshape((100,100))
        cv2.imwrite(save_path, img.astype(np.uint8))
        
    def compare_result(self, answer):
        result_2d = self.result.reshape((100,100))
        for i in range(100):
            for j in range(100):
                if answer[i,j] != result_2d[i,j]:
                    print("@", j,i, "Answer: ", answer[i,j], "DUT: ", result_2d[i,j])
                    print("Error Occured!")
                    break;

        print("END")
    
    def start(self, input_array):
        start_time = time.time()
        self.imem[0:10404] = input_array
        self.csr[1] = 1
        while True:
            if (self.csr[0] == 1):
                break
        self.result = self.omem[0:10000]
        end_time = time.time()
        runtime = end_time - start_time
        print(f"Runtime: {runtime*1000:.3f}ms")
        

In [ ]:
hw = PynqTestDriver('sobel_ps.bit')

In [ ]:
hw.start(input_data)

In [ ]:
hw.generate_img("result_img.jpg")

In [ ]:
result = hw.get_output(10000)

In [ ]:
# this command generate solution for sobel edge detector

from scipy.signal import convolve2d

sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=np.float32)

sobel_y = np.array([[-1,  -2,  -1],
                    [ 0,  0,  0],
                   [1,2,1]], dtype=np.float32)

grad_x = convolve2d(image, sobel_x, mode='valid')
grad_y = convolve2d(image, sobel_y, mode='valid')
abs_sobel_x = np.abs(grad_x)
abs_sobel_y = np.abs(grad_y)

sobel = abs_sobel_x + abs_sobel_y

grad_clipped = np.clip(sobel, 0, 255).astype(np.uint8)
cv2.imwrite('answer_image.png', grad_clipped)


In [ ]:
hw.compare_result(grad_clipped)

In [ ]:
# this command generate hex file for verilog simulation

def image_to_mem(image_path, mem_path):

    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img.shape != (102, 102):
        img = cv2.resize(img,(102,102))

    flat = img.flatten()

    hex_vals = [f"{val:02X}" for val in flat]

    with open(mem_path, 'w') as f:
        for val in hex_vals:
            f.write(val + '\n')

    print(f"[✓] Done for making Hex file for readmemh: {mem_path}")


image_to_mem("Penguin.jpg", "output_image.hex")